# Independent reproduction guide

This notebook is written for an external verifier who has downloaded two things:

1. the Code Availability repository; and
2. the separate reproduction data package listed in the Data Availability statement.

The workflow follows the manuscript logic: load the prepared bulk RNA matrix, optionally retrain the RNA-language model from that matrix, rerun manuscript analyses from frozen deposited artifacts, and display the key result panels from regenerated CSV source data.

The notebook displays figures in the notebook. It does not save publication figures.

## Step 1. Point the notebook to the code repository and data package

Change only `CODE_REPO`, `DATA_PACKAGE` and `WORK_DIR` if your directories are different.

`RUN_MODEL_TRAINING` controls the long full-data retraining route. Leave it as `False` when you only want to verify the deposited manuscript results from frozen artifacts; set it to `True` to train the RNA-language alignment and portrait-attention models from the prepared training matrix.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def find_code_repo() -> Path:
    env_path = os.environ.get('RNA_PORTRAIT_CODE_REPO')
    if env_path:
        candidate = Path(env_path).expanduser().resolve()
        if (candidate / 'rna_portrait').exists():
            return candidate
        raise FileNotFoundError(f'RNA_PORTRAIT_CODE_REPO does not look like the code repository: {candidate}')
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        candidate = candidate.resolve()
        if (candidate / 'rna_portrait').exists() and (candidate / 'workflows').exists():
            return candidate
    raise FileNotFoundError('Could not infer CODE_REPO. Set RNA_PORTRAIT_CODE_REPO to the code repository path.')


CODE_REPO = find_code_repo()
DATA_PACKAGE = Path(os.environ.get('RNA_PORTRAIT_DATA_PACKAGE', CODE_REPO.parent / 'nature_reproduction_data_package_20260611')).expanduser().resolve()
WORK_DIR = Path(os.environ.get('RNA_PORTRAIT_WORK_DIR', CODE_REPO.parent / 'rna_portrait_reproduction_run')).expanduser().resolve()

RUN_MODEL_TRAINING = False
RUN_MANUSCRIPT_PIPELINE = True
RUN_R_DECONVOLUTION = True

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'logs').mkdir(exist_ok=True)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print('CODE_REPO:', CODE_REPO)
print('DATA_PACKAGE:', DATA_PACKAGE)
print('WORK_DIR:', WORK_DIR)
print('RUN_MODEL_TRAINING:', RUN_MODEL_TRAINING)
print('RUN_MANUSCRIPT_PIPELINE:', RUN_MANUSCRIPT_PIPELINE)

## Step 2. Validate the expected files

This check makes sure the notebook is seeing the same release layout described in the Data Availability manifest.

In [ ]:
required_code_files = [
    CODE_REPO / 'rna_portrait' / '__init__.py',
    CODE_REPO / 'workflows' / 'run_training_pipeline.py',
    CODE_REPO / 'workflows' / 'run_manuscript_pipeline.py',
    CODE_REPO / 'workflows' / 'figures' / 'make_main_figures_and_source_data.py',
]
required_data_paths = [
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'expr_log.parquet',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'meta.csv',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'genes.npy',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'sample_ids.npy',
    DATA_PACKAGE / 'processed_data' / 'validation_profiles' / 'external_180',
    DATA_PACKAGE / 'processed_data' / 'validation_profiles' / 'multisource_450',
    DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_alignment_backbone' / 'bulk_multimodal_embedding.pt',
    DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_backbone_v8_topk64' / 'semantic_prototype_attention.pt',
    DATA_PACKAGE / 'source_data',
]
missing = [str(p) for p in required_code_files + required_data_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files or directories:' + chr(10) + chr(10).join(missing))
print('All required code and data paths are present.')

## Step 3. Inspect the training matrix

The expression table is stored as genes by samples. We first inspect metadata, gene symbols and sample identifiers, then read only a small expression preview to avoid loading the full matrix into notebook memory.

In [ ]:
TRAINING_MATRIX = DATA_PACKAGE / 'processed_data' / 'training_matrix'
meta = pd.read_csv(TRAINING_MATRIX / 'meta.csv')
genes = np.load(TRAINING_MATRIX / 'genes.npy', allow_pickle=True).astype(str)
sample_ids = np.load(TRAINING_MATRIX / 'sample_ids.npy', allow_pickle=True).astype(str)
summary = json.loads((TRAINING_MATRIX / 'summary.json').read_text())

print('Training summary:')
print(json.dumps(summary, indent=2)[:1200])
print('meta shape:', meta.shape)
print('n genes:', len(genes))
print('n sample ids:', len(sample_ids))
display(meta[['sample_id', 'source_dataset', 'feat_anatomical_site', 'feat_tumor_status', 'feat_disease_label', 'metadata_text']].head())
print('first genes:', genes[:10].tolist())
print('first sample ids:', sample_ids[:10].tolist())

In [ ]:
preview_sample_ids = sample_ids[:5].tolist()
expr_preview = pd.read_parquet(TRAINING_MATRIX / 'expr_log.parquet', columns=preview_sample_ids).head(10)
print('Expression preview shape:', expr_preview.shape)
display(expr_preview)

## Step 4. Inspect public-source and reproduction manifests

These manifests connect the notebook inputs to the Data Availability statement.

In [ ]:
public_manifest = pd.read_csv(DATA_PACKAGE / 'data_availability' / 'public_dataset_manifest.csv')
asset_manifest = pd.read_csv(DATA_PACKAGE / 'data_availability' / 'reproduction_asset_manifest.csv')
print('public source manifest:', public_manifest.shape)
display(public_manifest.head())
print('reproduction asset manifest:', asset_manifest.shape)
display(asset_manifest)

## Step 5. Optional full-data model training

This cell calls the repository training workflow on the prepared training matrix. It writes training artifacts to `WORK_DIR/trained_artifacts`.

The manuscript figures below use the frozen deposited artifacts in `DATA_PACKAGE/model_artifacts`, because fresh training may produce slightly different weights across hardware, PyTorch versions and random-number implementations. This separation is deliberate: retraining checks that the model is rebuildable; manuscript reproduction checks the exact reported results.

In [ ]:
TRAINED_ARTIFACT_ROOT = WORK_DIR / 'trained_artifacts'
training_cmd = [
    sys.executable, str(CODE_REPO / 'workflows' / 'run_training_pipeline.py'),
    '--training-matrix', str(TRAINING_MATRIX),
    '--artifact-root', str(TRAINED_ARTIFACT_ROOT),
    '--alignment-run-name', 'semantic_alignment_backbone',
    '--portrait-run-name', 'semantic_backbone_v8_topk64',
    '--skip-age-readout',
]
print(' '.join(training_cmd))
trained_summary_path = TRAINED_ARTIFACT_ROOT / 'bulk_multimodal_embedding' / 'semantic_backbone_v8_topk64' / 'summary.json'
if RUN_MODEL_TRAINING and trained_summary_path.exists():
    print('Training artifacts already exist; skipping retraining. Delete', TRAINED_ARTIFACT_ROOT, 'to rerun from scratch.')
elif RUN_MODEL_TRAINING:
    log_path = WORK_DIR / 'logs' / 'training_pipeline.log'
    env = os.environ.copy()
    env.setdefault('TOKENIZERS_PARALLELISM', 'false')
    with log_path.open('w', encoding='utf-8') as log:
        subprocess.run(training_cmd, cwd=str(CODE_REPO), stdout=log, stderr=subprocess.STDOUT, check=True, env=env)
    print('Training finished. Log:', log_path)
else:
    print('Training skipped. Set RUN_MODEL_TRAINING = True in Step 1 to run full-data retraining.')

In [ ]:
trained_summary_path = TRAINED_ARTIFACT_ROOT / 'bulk_multimodal_embedding' / 'semantic_alignment_backbone' / 'summary.json'
if trained_summary_path.exists():
    trained_summary = json.loads(trained_summary_path.read_text())
    print('Newly trained model summary:')
    print(json.dumps({k: v for k, v in trained_summary.items() if k != 'history'}, indent=2)[:2000])
    if 'history' in trained_summary:
        display(pd.DataFrame(trained_summary['history']).head())
else:
    frozen_summary_path = DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_alignment_backbone' / 'summary.json'
    frozen_summary = json.loads(frozen_summary_path.read_text())
    print('No newly trained summary found; showing deposited manuscript-training summary instead.')
    print(json.dumps({k: v for k, v in frozen_summary.items() if k != 'history'}, indent=2)[:2000])
    if 'history' in frozen_summary:
        display(pd.DataFrame(frozen_summary['history']).head())

## Step 6. Rerun manuscript analyses and source-data generation

This cell uses the deposited manuscript artifacts to regenerate the analysis tables and figure source data in `WORK_DIR`. It is the exact route used to check the manuscript's reported source-data tables.

In [ ]:
ANALYSIS_OUT = WORK_DIR / 'manuscript_analysis_tables'
FIGURE_OUT = WORK_DIR / 'figures'
R_LIB_ROOT = DATA_PACKAGE / 'software_environment' / 'r_libraries_deconvolution'

manuscript_cmd = [
    sys.executable, str(CODE_REPO / 'workflows' / 'run_manuscript_pipeline.py'),
    '--data-root', str(DATA_PACKAGE / 'processed_data'),
    '--artifact-root', str(DATA_PACKAGE / 'model_artifacts'),
    '--analysis-output-root', str(ANALYSIS_OUT),
    '--figure-output-root', str(FIGURE_OUT),
]
if RUN_R_DECONVOLUTION:
    manuscript_cmd += ['--r-lib-root', str(R_LIB_ROOT)]
else:
    manuscript_cmd += ['--skip-r-deconvolution']
print(' '.join(manuscript_cmd))
if RUN_MANUSCRIPT_PIPELINE:
    log_path = WORK_DIR / 'logs' / 'manuscript_pipeline.log'
    with log_path.open('w', encoding='utf-8') as log:
        subprocess.run(manuscript_cmd, cwd=str(CODE_REPO), stdout=log, stderr=subprocess.STDOUT, check=True)
    print('Manuscript pipeline finished. Log:', log_path)
else:
    print('Manuscript pipeline skipped. Set RUN_MANUSCRIPT_PIPELINE = True in Step 1 to regenerate tables and source data.')

## Step 7. Use the regenerated manuscript figure outputs

The cells below do not redraw simplified teaching plots. They display the same manuscript figures generated by `workflows/figures/make_main_figures_and_source_data.py`, so labels, legends, colours and axes remain synchronized with the manuscript HTML. Run Step 6 first when you want full independent reproduction.

In [ ]:
generated_source = FIGURE_OUT / 'source_data'
generated_figure_dir = FIGURE_OUT / 'figures'
reference_source = DATA_PACKAGE / 'source_data'

SOURCE_DATA = generated_source if generated_source.exists() else reference_source
FIGURE_IMAGE_DIR = generated_figure_dir

source_csvs = sorted(p for p in SOURCE_DATA.glob('*.csv') if not p.name.startswith('._'))
print('Using SOURCE_DATA:', SOURCE_DATA)
print('n source-data CSV files:', len(source_csvs))
print('first files:')
for p in source_csvs[:10]:
    print('-', p.name)

if not FIGURE_IMAGE_DIR.exists():
    raise FileNotFoundError(
        f'Manuscript figures were not found at {FIGURE_IMAGE_DIR}. '
        'Run Step 6 with RUN_MANUSCRIPT_PIPELINE = True to regenerate the same figures shown in the manuscript.'
    )
print('Using FIGURE_IMAGE_DIR:', FIGURE_IMAGE_DIR)

## Step 8. Display the regenerated manuscript figures

These are the publication-layout figures generated by the manuscript figure script. They are intentionally shown as complete figures rather than reimplemented panel-by-panel in the notebook, because the reproduction notebook should verify the manuscript outputs rather than introduce a second plotting style.

In [ ]:
from IPython.display import Image, display, Markdown

MANUSCRIPT_FIGURES = [
    ('Figure 1 | Architecture for RNA-to-language molecular portraiting', 'Figure_1_architecture.png'),
    ('Figure 2 | RNA profiles align with language-derived descriptions', 'Figure_2_RNA_language_alignment.png'),
    ('Figure 3 | Predefined disease names compress molecular portraits', 'Figure_3_portraits_not_single_labels.png'),
    ('Figure 4 | Independent biology supports portrait groups', 'Figure_4_biological_grounding.png'),
    ('Figure 5 | Mixing tests and source controls define the boundary', 'Figure_5_stress_tests_and_boundaries.png'),
    ('Extended Data 1 | Whole-profile and local-feature boundary analysis', 'Extended_Data_1_local_parts_boundary.png'),
]

missing = [name for _, name in MANUSCRIPT_FIGURES if not (FIGURE_IMAGE_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Missing regenerated manuscript figure files: ' + '; '.join(missing))

for title, filename in MANUSCRIPT_FIGURES:
    image_path = FIGURE_IMAGE_DIR / filename
    display(Markdown(f'### {title}'))
    display(Image(data=image_path.read_bytes(), format='png', width=980))

## Step 9. Inspect source data with reader-facing labels

The source CSV files retain internal machine-readable fields, but manuscript figures use reader-facing labels. This cell applies the same label vocabulary used by the manuscript figure code before displaying table previews.

In [ ]:
PORTRAIT_LABELS = {
    'stable_consensus': 'single clear signal',
    'hematologic_override': 'blood/immune',
    'epithelial_override': 'epithelial-like context',
    'clean_anchor_override': 'cleaner anchor context',
    'generic_context_override': 'broad context',
    'unsupported_semantics': 'weak evidence',
    'family_conflict': 'conflict',
    'other': 'other',
}
STATUS_LABELS = {
    'stable': 'single clear signal',
    'mixed': 'several signals',
    'unsupported': 'weak evidence',
}
DISPLAY_COLUMN_NAMES = {
    'semantic_state_family': 'molecular portrait family',
    'portrait_status': 'portrait status',
    'resolved_status': 'portrait status',
    'dominant_portrait_family': 'dominant portrait family',
    'closed_set_disease_family': 'predefined disease label',
}

def reader_label(value):
    text = str(value)
    return PORTRAIT_LABELS.get(text, STATUS_LABELS.get(text, text.replace('_', ' ')))

def reader_facing_table(df):
    preview = df.copy()
    for col in ['semantic_state_family', 'portrait_status', 'resolved_status', 'dominant_portrait_family']:
        if col in preview.columns:
            preview[col] = preview[col].map(reader_label)
    if 'closed_set_disease_family' in preview.columns:
        preview['closed_set_disease_family'] = preview['closed_set_disease_family'].map(lambda x: str(x).replace('_', ' '))
    rename_cols = {c: reader_label(c) for c in preview.columns if c in PORTRAIT_LABELS}
    rename_cols.update({c: DISPLAY_COLUMN_NAMES[c] for c in preview.columns if c in DISPLAY_COLUMN_NAMES})
    return preview.rename(columns=rename_cols)

def display_source_table(filename, title, max_rows=8):
    path = SOURCE_DATA / filename
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    display(Markdown(f'### {title}'))
    display(reader_facing_table(df).head(max_rows))

# These previews correspond to the figures above and expose the same quantitative source data.
display_source_table('figure_2_cosine_summary.csv', 'Figure 2b source data: RNA-text cosine')
display_source_table('figure_3_status_summary.csv', 'Figure 3b source data: portrait-status fractions')
display_source_table('figure_3_predefined_label_portraits.csv', 'Figure 3d source data: disease labels by portrait family')
display_source_table('figure_4_marker_heatmap_matrix.csv', 'Figure 4a source data: marker programmes by portrait family')
display_source_table('figure_5_partial_r2.csv', 'Figure 5c source data: portrait signal after controls')

## Step 10. Confirm manuscript figure files and source-data files

This check records the exact regenerated figure files and source-data tables that the notebook is displaying. The expected figure files are the same filenames referenced by the manuscript HTML.

In [ ]:
figure_rows = []
for title, png_name in MANUSCRIPT_FIGURES:
    stem = Path(png_name).stem
    row = {'figure': title, 'png': str(FIGURE_IMAGE_DIR / png_name)}
    for ext in ['svg', 'pdf', 'tiff']:
        row[ext] = str(FIGURE_IMAGE_DIR / f'{stem}.{ext}')
        row[f'{ext}_exists'] = (FIGURE_IMAGE_DIR / f'{stem}.{ext}').exists()
    row['png_exists'] = (FIGURE_IMAGE_DIR / png_name).exists()
    figure_rows.append(row)
figure_manifest = pd.DataFrame(figure_rows)
display(figure_manifest)

source_manifest_path = SOURCE_DATA / 'source_data_manifest.csv'
if source_manifest_path.exists():
    source_manifest = pd.read_csv(source_manifest_path)
    display(source_manifest.head(12))
else:
    print('No source_data_manifest.csv found in', SOURCE_DATA)

## Step 11. Verify that deposited and regenerated source-data tables agree

If Step 6 was run, this cell compares newly generated figure source-data CSVs against the deposited reference CSVs. Numeric source-data tables should match. Text-only provenance fields may differ when paths are regenerated on a different machine.

In [ ]:
reference_source = DATA_PACKAGE / 'source_data'
if generated_source.exists():
    rows = []
    ref_files = sorted(p for p in reference_source.glob('*.csv') if not p.name.startswith('._'))
    for ref in ref_files:
        new = generated_source / ref.name
        if not new.exists():
            rows.append({'file': ref.name, 'status': 'missing_new'})
            continue
        a = pd.read_csv(ref)
        b = pd.read_csv(new)
        same_shape = a.shape == b.shape
        same_columns = list(a.columns) == list(b.columns)
        numeric_cols = [c for c in a.columns if c in b.columns and pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c])]
        max_abs = 0.0
        for c in numeric_cols:
            diff = (a[c].astype(float) - b[c].astype(float)).abs().max()
            if pd.notna(diff):
                max_abs = max(max_abs, float(diff))
        exact_text_match = same_shape and same_columns and a.astype(str).equals(b.astype(str))
        status = 'match' if exact_text_match else ('numeric_match_text_check' if same_shape and same_columns and max_abs == 0.0 else 'check')
        rows.append({
            'file': ref.name,
            'status': status,
            'same_shape': same_shape,
            'same_columns': same_columns,
            'max_abs_numeric_difference': max_abs,
        })
    comparison = pd.DataFrame(rows)
    display(comparison)
else:
    print('No regenerated source-data directory found. Run Step 6 with RUN_MANUSCRIPT_PIPELINE=True to compare regenerated CSVs.')